# API

## Class: `SpaMOMA`

`SpaMOMA` provides an end-to-end framework for spatial multi-omics slice alignment, translation, clustering, and resolution enhancement.  
It supports omics data (RNA, ATAC, protein, metabolite) as well as image inputs.

---

## `__init__(self, save_dir)`

Initialize a SpaMOMA project.

### Parameters

- **save_dir** (`str`)  
  Directory where all intermediate files, trained models, logs, and alignment outputs will be stored.

### Description

- Creates the project directory if it does not exist.
- Initializes:
  - `input_slice_dict`: dictionary storing slice metadata.
  - `reference_sample_id`: default reference slice.
  - `coarse_epochs` / `fine_epochs`: training configuration placeholders.

---

## `add_slice(self, sample_id, raw_data_path, feature)`

Register a slice into the SpaMOMA workflow.

### Parameters

- **sample_id** (`str`)  
  Unique identifier for the slice.

- **raw_data_path** (`str`)  
  Path to the raw input file:
  - `.h5ad` for omics data
  - image file for `img` feature

- **feature** (`str`)  
  Type of input data. Must be one of:
  - `'rna'`
  - `'gene_activity_score'`
  - `'peak'`
  - `'protein'`
  - `'metabolite'`
  - `'img'`

### Description

- Adds slice metadata to `input_slice_dict`.
- The first added slice is automatically set as the reference slice.

---

## `set_reference(self, sample_id)`

Set the reference slice for alignment.

### Parameters

- **sample_id** (`str`)  
  Identifier of the slice to use as reference.

### Description

- Defines the coordinate system to which all other slices will be aligned.

---

## `preprocess_input_slices(self, params_dict={})`

Preprocess all registered slices.

### Parameters

- **params_dict** (`dict`, optional)  
  Dictionary specifying per-slice preprocessing parameters.

For omics slices:
- `svf_num` – Number of spatially variable features.
- `hvf_num` – Number of highly variable features.
- `pseudo_color_key` – Key for storing pseudo-color embedding.
- `graph_mode` – `'KNN'` or `'Radius'`.
- `rad_cutoff` – Radius threshold (if using Radius graph).
- `k_cutoff` – KNN graph neighbor number (if using KNN graph).

For image slices:
- `downsample` (`bool`) – Whether to apply color quantization.
- `n_clusters` – Number of color clusters.
- `gaussian_kernel` – Gaussian smoothing kernel.

### Description

- Omics slices:
  - Feature selection (HVG + SVG)
  - Normalization & log transformation
  - STAGATE embedding
  - PCA to 3D pseudo-color space
  - Saves processed `.h5ad`

- Image slices:
  - Optional color quantization

---

## `generate_imgs(self, params_dict={})`

Generate Spatial Pattern Images (SPI).

### Parameters

- **params_dict** (`dict`, optional)  
  Per-slice visualization parameters:
  - `max_distance_ratio`
  - `interpolation_method`
  - `gaussian_sigma`

### Description

- For omics slices:
  - Converts pseudo-color embeddings into RGB spatial images.
- For image slices:
  - Saves original or downsampled image.
- Stores SPI path for downstream training.

---

## `train(...)`

Train coarse and fine alignment models.

### Parameters

- **coarse_epochs** (`int`) – Coarse training epochs.
- **fine_epochs** (`int`) – Fine training epochs.
- **batch_size** (`int`)
- **parallel_mode** (`bool`) – Run coarse and fine training in parallel.
- **coarse_checkpoint_save_step** (`int`, optional)
- **fine_checkpoint_save_step** (`int`, optional)
- **coarse_gpu** (`int`) – GPU ID for coarse stage.
- **fine_gpu** (`int`) – GPU ID for fine stage.

### Description

Two-stage training:

1. **Coarse alignment**
   - Multi-angle rotation augmentation
   - Progressive angle expansion

2. **Fine alignment**
   - Single-angle refinement

- Launches subprocesses for adaptation.
- Saves logs to:
  - `coarse_adapt.log`
  - `fine_adapt.log`

---

## `align_slices(self, coarse_weights_path=None, fine_weights_path=None, visualize_correspondences=False)`

Perform slice alignment.

### Parameters

- **coarse_weights_path** (`str`, optional) - Path to pretrained weights for the coarse matching network (RoMa). If None, it will automaticlly scan weigths file under the weights directory.
- **fine_weights_path** (`str`, optional) -Path to pretrained weights for the fine matching network. 
- **visualize_correspondences** (`bool`) -Whether to visualize coarse and fine matching correspondences during alignment.

### Description

Pipeline:

1. Load reference & source SPIs
2. Coarse matching (RoMa)
3. Estimate affine transformation
4. Fine matching
5. Compose transformations
6. Warp images or spatial coordinates

Outputs saved in:
  - `save_dir/alignment_output/`
  - `.png` for images
  - `.h5ad` for omics

---

## `read_aligned_data(self, sample_id)`

Load aligned result.

### Parameters

- **sample_id** (`str`) - Identifier of the aligned sample to load. Must match the previous input.

### Returns

- `numpy.ndarray` (if image)
- `AnnData` (if omics)

### Description

Reads aligned data from `alignment_output`.

---

## `visualize_aligned_slices(...)`

Visualize overlaid aligned slices.

### Parameters

- **figsize** (`tuple`) - Size of the matplotlib figure.
- **point_size** (`int`) - Marker size when plotting spatial spots.
- **alpha_adata** (`float`) - Transparency of omics spot visualization.
- **alpha_img** (`float`) - Transparency of background image overlay.
- **color_map** (`dict`, optional) - Dictionary mapping domain labels or clusters to colors. If None, default color palette is used.
- **save_fig** (`bool`) - Whether to save the visualization figure.
- **dpi** (`int`) - Resolution (dots per inch) when saving the figure.

### Description

- Overlays aligned slices on reference.
- Supports both images and spatial omics.
- Saves figure optionally.

---

## `SpaMOMA_translate(self, adata_target, adata_src, save_path, k_neighbors=5, overlap_distance_threshold=20)`

Perform cross-slice modality translation.

### Parameters

- **adata_target** (`AnnData`) - Target slice to receive transferred features.
- **adata_src** (`AnnData`) - Source slice providing features for translation.
- **save_path** (`str`) - Path to save translated AnnData object.
- **k_neighbors** (`int`) - Number of nearest spatial neighbors used for feature aggregation.
- **overlap_distance_threshold** (`float`) - Maximum spatial distance to consider two spots overlapping.

### Description

SpaMOMA performs translation differently for **overlapping** and **non-overlapping** regions.

#### 1️⃣ Overlapping Regions

For spatially overlapping spots between the two slices:

- Direct cross-slice feature aggregation is performed.
- For each target spot, features from spatially matched source neighbors are aggregated.
- Aggregation is based on spatial proximity and nearest-neighbor search.


#### 2️⃣ Non-Overlapping Regions

For target spots outside the overlapping region:

- Direct spatial correspondence is unavailable.
- SpaMOMA computes feature similarity between:
  - Non-overlapping target spots
  - Overlapping target spots (which already have translated source features)

- The source features of overlapping spots are then propagated to non-overlapping spots based on similarity in the target modality feature space.

In other words:

> Non-overlapping regions are predicted using similarity-weighted propagation from overlapping regions.


---

## `SpaMOMA_clustering(...)`

Perform integrative clustering for paired modalities.

### Parameters

- **adata1**, **adata2** (`AnnData`) - Two paired modalities (e.g., RNA & ATAC).
- **modality1**, **modality2** (`str`) - Names of modalities, used for embedding routing.
- **data_type** (`str`) - Specifies dataset type (e.g., "Spatial-epigenome-transcriptome"), determines preprocessing and embedding strategy.
- **n_domains** (`int`) - Number of clusters/domains to identify.
- **method_list** (`list`) - Multimodal embedding methods to compute. 
  Supported:
  - `'SpatialGlue'`
  - `'MISO'`
  - `'PRESENT'`

- **cluster_types** (`list`) - Clustering algorithms applied to embeddings.
  Supported:
  - `'leiden'`
  - `'louvain'`
  - `'mclust'`

- **add_key** (`str`) - Key name under adata.obs where clustering results will be stored.

### Description

SpaMOMA performs multi-method integrative clustering using a consensus strategy.

#### Step 1: Multimodal Embedding

For each method in `method_list`, a multimodal embedding is computed independently.

#### Step 2: Clustering per Embedding

For each generated embedding, clustering is performed using all algorithms specified in `cluster_types`.

Therefore, the total number of clustering results is:

\[
\text{Total clusterings} = |\text{method_list}| \times |\text{cluster_types}|
\]

Each embedding × clustering algorithm combination produces one clustering result.

#### Step 3: Consensus Voting

All clustering results are integrated using a consensus voting strategy:

- Each spot receives multiple cluster labels.
- A voting mechanism aggregates these labels.
- The final domain assignment is determined based on agreement across methods.

### Output

- Returns clustered `AnnData`
- Stores final consensus clustering under `adata.obs[add_key]`

---

## `SpaMOMA_clustering_multiple_slices(...)`

Joint clustering across two aligned slices.

### Parameters

- **multiple_slices_dict** (`dict`) - Dictionary of slice_name → adata.
- **reference_modality_dict** (`dict`) - Specifies which modality serves as reference per slice for label prediction of non-overlap region.
- **batch_key** (`str`) - Metadata key identifying slice origin.
- **n_domains** (`int`) - Number of shared domains across slices.
- **spatial_key** (`str`) - Key storing spatial coordinates.
- **add_key** (`str`) - Key for storing joint clustering results.
- **overlap_distance_threshold** (`float`) - Spatial threshold for defining overlapping regions.
- **smooth** (`bool`) - Whether to apply spatial smoothing.
- **k_smooth** (`int`) - Number of neighbors used for smoothing.
- **smooth_outlier_threshold** (`float`) - Threshold for identifying outlier spots during smoothing.

### Description

SpaMOMA performs joint clustering using different strategies for overlapping and non-overlapping regions.


#### 1️⃣ Overlapping Regions

- Spatially overlapping spots between slices are identified using `overlap_distance_threshold`.
- Only overlapping regions are used to compute a joint multimodal embedding (via PRESENT).
- Clustering is performed in this shared embedding space to identify cross-slice shared domains.

This ensures domain consistency in spatially aligned regions.

#### 2️⃣ Non-Overlapping Regions

For spots outside the overlapping region:

- Joint embedding is not directly available.
- For each slice, non-overlapping spots are compared to overlapping spots **within the same slice** using the specified `reference_modality_dict`.
- Similarity is computed in the reference modality feature space.
- Domain labels from overlapping spots are propagated to non-overlapping spots based on similarity.

In other words:

> Non-overlapping labels are predicted via similarity-based label transfer from overlapping regions within each slice.


#### 3️⃣ Optional Spatial Smoothing

If `smooth=True`:

- A k-nearest neighbor spatial graph is constructed.
- Local label refinement is performed.
- Outliers are corrected using `smooth_outlier_threshold`.


### Output

- Returns updated `AnnData` objects
- Stores joint clustering results under `adata.obs[add_key]`

---

## `SpaMOMA_enhance_resolution(self, img_path, adata, pseudo_spot_size=10, weight_decay=4, n_neighbors=4)`

Enhance spot resolution to pixel-level resolution.

### Parameters

- **img_path** (`str`) - Path to high-resolution histology image.
- **adata** (`AnnData`) - Original low-resolution spatial omics data.
- **pseudo_spot_size** (`int`) - Diameter (in pixels) of generated pseudo-spots.
- **weight_decay** (`float`) - Controls spatial influence decay when aggregating neighboring signals.
- **n_neighbors** (`int`) - Number of neighboring spots used during enhancement.

### Description

- Generates pseudo-spots.
- Uses spatial and image cues.
- Produces enhanced-resolution AnnData.

### Returns

- Enhanced `AnnData`

---


